In [37]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from nltk.translate.meteor_score import single_meteor_score
from nltk.tokenize import word_tokenize
import numpy as np

import torch

In [40]:
model_name = "humarin/chatgpt_paraphraser_on_T5_base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [3]:
splits = {
    'train': 'synthetic_text_to_sql_train.snappy.parquet',
    'test': 'synthetic_text_to_sql_test.snappy.parquet'
}
train_df = pd.read_parquet("hf://datasets/gretelai/synthetic_text_to_sql/" + splits["train"])

In [8]:
train_df.columns

Index(['id', 'domain', 'domain_description', 'sql_complexity',
       'sql_complexity_description', 'sql_task_type',
       'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql',
       'sql_explanation'],
      dtype='object')

In [4]:
train_df = train_df[["sql","sql_prompt","sql_context"]]

In [12]:
def paraphrase(text, num_return_sequences=1):
    inputs = tokenizer.encode(f"paraphrase: {text}", return_tensors="pt", max_length=256, truncation=True)
    outputs = model.generate(
        inputs,
        max_length=256,
        num_return_sequences=num_return_sequences,
    )
    if num_return_sequences == 1:
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

In [6]:
new_df = train_df.copy()
new_rows = []

In [31]:
test = pd.DataFrame(new_df).iloc[:100]['sql_prompt']

In [24]:
def get_meteors(text):
    meteors = []
    for row in text:
        cur_paraphrase = paraphrase(row)
        cur_meteor = single_meteor_score(word_tokenize(row), word_tokenize(cur_paraphrase))
        meteors.append(cur_meteor)
    return meteors

In [32]:
meteors = get_meteors(test)

In [33]:
median_meteor = np.median(meteors)

In [34]:
min_meteor = median_meteor*0.75
max_meteor = median_meteor*1.25